In [13]:
import os, IPython
from openai import OpenAI

from dotenv import load_dotenv

load_dotenv()

# print(os.getenv('OPENAI_API_KEY'))

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

In [6]:
def get_completion(messages, model='gpt-5.1', temperature=0, max_tokens=300):
    
    response = client.chat.completions.create(
        model=model,
        messages = messages,
        temperature = temperature,
        max_tokens = max_tokens,)
    
    # response.choices[0].message["content"]
    return response

## Be specific and clear 

write instructions as clear as possible to get desired LLM behaviors 

In [7]:
global_trending_movies = ['The suicide squad', 'Animal', '12th fail']

system_message = """
your task is to recommend movies to the customer

you are responsible to recommend a movie from the top global trending movies from {global_trending_movies}

you should refrain from asking users for their preferences and avoid asking personal information

If you dont have a movie to recommend or dont know the user interests, you should respond "Sorry, couldnt find a movie recommend today"

"""


user_request = """
Please recommend a movie
"""

message = [
    
    {
        "role": "system",
        "content": system_message.format(global_trending_movies=global_trending_movies)
    },
    {
        "role": "user",
        "content":user_request
    }
]


response = get_completion(message)

print(response)


BadRequestError: Error code: 400 - {'error': {'message': "Unsupported parameter: 'max_tokens' is not supported with this model. Use 'max_completion_tokens' instead.", 'type': 'invalid_request_error', 'param': 'max_tokens', 'code': 'unsupported_parameter'}}

## Add delimiters

Adding delimiters to help better structure instructions and overall prompt components. This is beneficial to get more reliable responses

In [ ]:
user_request = """

Conver the following code block within ### <code> ### to python

###
strings.push('one')
strings.push('two')
strings.push('three')
###
"""

message = [{
    "role":"user",
    "content":user_request
}]

# resp = get_completion(message)

## Specify output format 

If format of responses are important, then explicitly state it in the prompt to get desired results

In [ ]:
user_request = """
your task is: given the product description, return the requested information in the section delimited by ### ###. 
Format the output as JSON

Product Description: Introducting Nike Air Max 270 react : a comfortable and stylish snearkers that combines two of Nike's best technology


###
product_name: the name of the product
product_brand: the name of the brand

###
"""



message = [
    {
        "role": "user",
        "content": user_request
    }
]


## Think step by step 

In [ ]:
user_request = """ Check if the odd numbers in the group add up to an even number : 15,32,27,4,6,8
                
solve by breaking the problem into steps. First identify the odd numbers, add them and indicate whether the result is odd or even

"""

## Breaking into simpler and smaller subtasks - Prompt chaining

## Role playing

In [ ]:
system_message = """

The following is a conversation with an AI reserch assistant. The assistant tone is technical and scientific
"""

user_message_1 = """
Hello, who are you?
"""

ai_message_1 = """
Greeting! I am an AI research assistant. How can i help you today?
"""

prompt = """

Human: Can you tell me about the creation of the blackhole
"""


message = [
    {
        "role":"system",
        "content": system_message
    },
    {
        "role":"user",
        "content": user_message_1
    },
    {
        "role":"assistant",
        "content":ai_message_1
    },
    {
        "role":"user",
        "content": prompt
    }
]

## Few-shot In-Context learning

provide the model some examples to better steer it at the task you are interested in. Make sure the examples are diverse

In [ ]:
prompt = """

your task is to classify the input text delimited by ``` as either offensive or non-offensive

Text: I love you
Output: non-offensive

Text: I dislike all the people working here
Output: offensive

Text: I think this feature is not ideal
Output: non-offensive

Text: These people are stupi
Output: offensive


Text: {user_input}
output: 
"""

message = [{
    "role": "user",
    "content": prompt.format(user_input="```I respectfully disagree with your opinion.```")
}]


## Chain of Thought

In [ ]:
system_message="""
your task is to make movie recommendation based on user request (delimited by ```)

step 1: check if user is asking about movies. If not say "ask something about movies"
step 2: if asking about movies check if there are any interests 
step 3: check if there are any movies that we can recommend from following list {movies}
step 4: Prepare a friendly response about movie recommendations. Reponse should be in below format 


step 1 : <step 1 reasoning>
step 2 : <step 2 reasoning>
step 3 : <step 3 reasoning>
step 4 : <final response>

"""


message = [   
    {
        "role": "system"
        "conent": system_message.format(movies=movies)
    },
    {
        "role": "user"
        "content":"```Do you have any drama movies```"
    }   
]

movie_recom_resp = get_completion(message)

## prompt chaining

In [ ]:
system_message = """
 
 you will be given a list of steps that model has responded with. your task is to extract and respond with text in step 4
 from following text : {movie_recom_resp}
 
"""

message = [
    {
        "role": "system"
        "content": system.format(movie_recom_resp=movie_recom_resp)
    }
]

## ReAct

Prompting framework.
Combines LLMs with external tools like APIs, Databases

## RAG

In [15]:
"""
Tweet Sentiment Analyzer using OpenAI API with Few-Shot Prompting
Processes financial tweets and extracts sentiment, key entities, and summaries.
"""

import csv
import json
import os
import time
from typing import Optional
from openai import OpenAI


# Few-shot examples extracted from processed_financial_tweets.csv
FEW_SHOT_EXAMPLES = [
    {
        "tweet": "*EU SET TO MEET ON RETALIATION PLANS AS US TRADE STANCE HARDENS",
        "output": {
            "Date": "2025-07-20",
            "tweet": "*EU SET TO MEET ON RETALIATION PLANS AS US TRADE STANCE HARDENS",
            "key_entities": "EU, US",
            "new_summary": "EU plans retaliation meeting amid hardening US trade stance.",
            "sentiment_score": 1,
            "reasoning": "Escalating trade tensions and retaliation plans are negative for markets."
        }
    },
    {
        "tweet": "DONALD TRUMP AND XI JINPING TIPPED TO MEET AHEAD OF OR DURING APEC SUMMIT IN SOUTH KOREA- SCMP",
        "output": {
            "Date": "2025-07-20",
            "tweet": "DONALD TRUMP AND XI JINPING TIPPED TO MEET AHEAD OF OR DURING APEC SUMMIT IN SOUTH KOREA- SCMP",
            "key_entities": "Donald Trump, Xi Jinping, US, China, APEC, South Korea",
            "new_summary": "Trump and Xi expected to meet near APEC summit.",
            "sentiment_score": 4,
            "reasoning": "High-level talks between US/China leaders usually signal potential conflict resolution."
        }
    },
    {
        "tweet": "$COIN - CANTOR FITZGERALD RAISES COINBASE PRICE TARGET TO $500 FROM $292...",
        "output": {
            "Date": "2025-07-21",
            "tweet": "$COIN - CANTOR FITZGERALD RAISES COINBASE PRICE TARGET TO $500 FROM $292...",
            "key_entities": "$COIN, Coinbase, Cantor Fitzgerald",
            "new_summary": "Cantor Fitzgerald raises Coinbase PT to $500, citing stablecoin/L2 growth.",
            "sentiment_score": 5,
            "reasoning": "Major price target hike and positive fundamental analysis."
        }
    },
    {
        "tweet": "$DELL - DELL TECHNOLOGIES CONFIRMS BREACH OF TEST LAB PLATFORM BY WORLD LEAKS EXTORTION GROUP - BLEEPING COMPUTER",
        "output": {
            "Date": "2025-07-21",
            "tweet": "$DELL - DELL TECHNOLOGIES CONFIRMS BREACH OF TEST LAB PLATFORM BY WORLD LEAKS EXTORTION GROUP - BLEEPING COMPUTER",
            "key_entities": "$DELL, Dell Technologies",
            "new_summary": "Dell confirms breach of test lab platform.",
            "sentiment_score": 1,
            "reasoning": "Confirmed security breach is negative news."
        }
    },
    {
        "tweet": "OVER $1 TRILLION ERASED FROM U.S. STOCK MARKET AMID TRUMP TARIFF TURMOIL WEAK JOBS DATA",
        "output": {
            "Date": "2025-08-01",
            "tweet": "OVER $1 TRILLION ERASED FROM U.S. STOCK MARKET AMID TRUMP TARIFF TURMOIL WEAK JOBS DATA",
            "key_entities": "US Stock Market, Trump, Tariffs, Jobs Data",
            "new_summary": "Market loses $1T due to tariff turmoil and weak jobs.",
            "sentiment_score": 0,
            "reasoning": "Major wealth destruction and verified bearish catalyst."
        }
    }
]


def build_few_shot_prompt(tweet: str, date: str) -> list:
    """
    Build the few-shot prompt messages for OpenAI API.
    
    Args:
        tweet: The tweet to analyze
        date: The date of the tweet
        
    Returns:
        List of message dictionaries for OpenAI chat completion
    """
    system_message = """You are a financial analyst and expert in stock market trading.
For each tweet, your task is as follows:
1. Extract key entities from the tweet (stocks, companies, people, organizations, events).
2. Generate a new summary of the tweet based on extracted entities.
3. Analyze the sentiment of the tweet based on new summary with extracted entities and give it a score from 0-5:
   - 0: Extremely bearish
   - 1: Bearish
   - 2: Slightly bearish / Neutral-negative
   - 3: Neutral
   - 4: Bullish
   - 5: Extremely bullish

IMPORTANT: Respond ONLY with valid JSON. No additional text or explanation outside the JSON.
The JSON must have these exact keys: Date, tweet, key_entities, new_summary, sentiment_score, reasoning"""

    messages = [{"role": "system", "content": system_message}]
    
    # Add few-shot examples
    for example in FEW_SHOT_EXAMPLES:
        # User message (the tweet to analyze)
        messages.append({
            "role": "user",
            "content": f"Analyze this tweet:\n{example['tweet']}"
        })
        # Assistant response (the expected output)
        messages.append({
            "role": "assistant",
            "content": json.dumps(example['output'], indent=2)
        })
    
    # Add the actual tweet to analyze
    messages.append({
        "role": "user",
        "content": f"Analyze this tweet (Date: {date}):\n{tweet}"
    })
    
    return messages


def analyze_tweet(client: OpenAI, tweet: str, date: str, max_retries: int = 3) -> Optional[dict]:
    """
    Call OpenAI API to analyze a tweet.
    
    Args:
        client: OpenAI client instance
        tweet: The tweet text to analyze
        date: The date of the tweet
        max_retries: Maximum number of retry attempts
        
    Returns:
        Dictionary with analysis results or None if failed
    """
    messages = build_few_shot_prompt(tweet, date)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",  # Cost-effective model; change to "gpt-4o" for better quality
                messages=messages,
                temperature=0.3,  # Lower temperature for more consistent outputs
                max_tokens=500,
                response_format={"type": "json_object"}  # Enforce JSON response
            )
            
            # Parse the response
            content = response.choices[0].message.content
            result = json.loads(content)
            
            # Ensure all required fields are present
            required_fields = ["Date", "tweet", "key_entities", "new_summary", "sentiment_score", "reasoning"]
            for field in required_fields:
                if field not in result:
                    result[field] = ""
            
            # Override Date with the input date to ensure consistency
            result["Date"] = date
            result["tweet"] = tweet
            
            return result
            
        except json.JSONDecodeError as e:
            print(f"  JSON parsing error on attempt {attempt + 1}: {e}")
            if attempt < max_retries - 1:
                time.sleep(1)
        except Exception as e:
            print(f"  API error on attempt {attempt + 1}: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
    
    return None


def process_tweets(
    input_file: str,
    output_file: str,
    rate_limit_delay: float = 0.5
):
    """
    Process tweets from input CSV and write analysis results to output CSV.
    
    Args:
        input_file: Path to input CSV (Date, Time, Description columns)
        output_file: Path to output CSV (pipe-delimited)
        api_key: OpenAI API key (uses OPENAI_API_KEY env var if not provided)
        rate_limit_delay: Delay between API calls in seconds
    """
    # Initialize OpenAI client
#     api_key = api_key or os.getenv("OPENAI_API_KEY")
#     if not api_key:
#         raise ValueError("OpenAI API key not found. Set OPENAI_API_KEY environment variable or pass api_key parameter.")
    
#     client = OpenAI(api_key=api_key)
    
    # Read input CSV
    tweets_to_process = []
    with open(input_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Handle different possible column names
            date = row.get('Date') or row.get('date') or ''
            time_val = row.get('Time') or row.get('time') or ''
            tweet = row.get('Description') or row.get('description') or row.get('tweet') or row.get('Tweet') or ''
            
            if tweet.strip():
                tweets_to_process.append({
                    'date': date,
                    'time': time_val,
                    'tweet': tweet.strip()
                })
    
    print(f"Found {len(tweets_to_process)} tweets to process")
    
    # Process tweets and collect results
    results = []
    for i, item in enumerate(tweets_to_process, 1):
        print(f"Processing tweet {i}/{len(tweets_to_process)}...")
        
        result = analyze_tweet(client, item['tweet'], item['date'])
        
        if result:
            results.append(result)
            print(f"  Sentiment Score: {result.get('sentiment_score', 'N/A')}")
        else:
            # Create a fallback result for failed analyses
            results.append({
                "Date": item['date'],
                "tweet": item['tweet'],
                "key_entities": "ERROR",
                "new_summary": "Failed to analyze",
                "sentiment_score": -1,
                "reasoning": "API call failed after retries"
            })
            print("  Failed to analyze tweet")
        
        # Rate limiting
        if i < len(tweets_to_process):
            time.sleep(rate_limit_delay)
    
    # Write output CSV (pipe-delimited)
    output_fields = ["Date", "tweet", "key_entities", "new_summary", "sentiment_score", "reasoning"]
    
    with open(output_file, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=output_fields, delimiter='|', quoting=csv.QUOTE_ALL)
        writer.writeheader()
        
        for result in results:
            # Ensure all fields are strings and handle any nested structures
            row = {}
            for field in output_fields:
                value = result.get(field, '')
                # Convert lists/dicts to string if needed
                if isinstance(value, (list, dict)):
                    value = json.dumps(value)
                # Clean up any pipe characters in the data
                if isinstance(value, str):
                    value = value.replace('|', ',')
                row[field] = value
            writer.writerow(row)
    
    print(f"\nProcessing complete! Results written to: {output_file}")
    print(f"Total tweets processed: {len(results)}")
    
    # Print summary statistics
    valid_scores = [r['sentiment_score'] for r in results if isinstance(r.get('sentiment_score'), (int, float)) and r['sentiment_score'] >= 0]
    if valid_scores:
        avg_score = sum(valid_scores) / len(valid_scores)
        print(f"Average sentiment score: {avg_score:.2f}")
        print(f"Bullish tweets (4-5): {len([s for s in valid_scores if s >= 4])}")
        print(f"Neutral tweets (2-3): {len([s for s in valid_scores if 2 <= s <= 3])}")
        print(f"Bearish tweets (0-1): {len([s for s in valid_scores if s <= 1])}")
        

def main():
    """Main entry point."""
#     import argparse
    
#     parser = argparse.ArgumentParser(description='Analyze financial tweets for sentiment using OpenAI API')
#     parser.add_argument('input_file', help='Input CSV file (Date, Time, Description columns)')
#     parser.add_argument('output_file', help='Output CSV file (pipe-delimited)')
#     parser.add_argument('--api-key', help='OpenAI API key (or set OPENAI_API_KEY env var)')
#     parser.add_argument('--delay', type=float, default=0.5, help='Delay between API calls in seconds')
    
#     args = parser.parse_args()
    
    process_tweets(
        input_file='/Users/rc/Downloads/walter - walter_extract_v1.csv',
        output_file='/Users/rc/Downloads/walter_extract_op.csv',
    )



    
main()

Found 9109 tweets to process
Processing tweet 1/9109...
  Sentiment Score: 1
Processing tweet 2/9109...
  Sentiment Score: 3
Processing tweet 3/9109...
  Sentiment Score: 4
Processing tweet 4/9109...
  Sentiment Score: 4
Processing tweet 5/9109...
  Sentiment Score: 1
Processing tweet 6/9109...
  Sentiment Score: 4
Processing tweet 7/9109...
  Sentiment Score: 3
Processing tweet 8/9109...
  Sentiment Score: 3
Processing tweet 9/9109...
  Sentiment Score: 5
Processing tweet 10/9109...
  Sentiment Score: 1
Processing tweet 11/9109...
  Sentiment Score: 1
Processing tweet 12/9109...
  Sentiment Score: 3
Processing tweet 13/9109...
  Sentiment Score: 3
Processing tweet 14/9109...
  Sentiment Score: 3
Processing tweet 15/9109...
  Sentiment Score: 3
Processing tweet 16/9109...
  Sentiment Score: 3
Processing tweet 17/9109...
  Sentiment Score: 3
Processing tweet 18/9109...
  Sentiment Score: 4
Processing tweet 19/9109...
  Sentiment Score: 3
Processing tweet 20/9109...
  Sentiment Score: 3


  Sentiment Score: 4
Processing tweet 167/9109...
  Sentiment Score: 2
Processing tweet 168/9109...
  Sentiment Score: 1
Processing tweet 169/9109...
  Sentiment Score: 3
Processing tweet 170/9109...
  Sentiment Score: 4
Processing tweet 171/9109...
  Sentiment Score: 4
Processing tweet 172/9109...
  Sentiment Score: 2
Processing tweet 173/9109...
  Sentiment Score: 4
Processing tweet 174/9109...
  Sentiment Score: 4
Processing tweet 175/9109...
  Sentiment Score: 3
Processing tweet 176/9109...
  Sentiment Score: 3
Processing tweet 177/9109...
  Sentiment Score: 2
Processing tweet 178/9109...
  Sentiment Score: 2
Processing tweet 179/9109...
  Sentiment Score: 2
Processing tweet 180/9109...
  Sentiment Score: 3
Processing tweet 181/9109...
  Sentiment Score: 4
Processing tweet 182/9109...
  Sentiment Score: 3
Processing tweet 183/9109...
  Sentiment Score: 4
Processing tweet 184/9109...
  Sentiment Score: 4
Processing tweet 185/9109...
  Sentiment Score: 1
Processing tweet 186/9109...


  Sentiment Score: 4
Processing tweet 331/9109...
  Sentiment Score: 3
Processing tweet 332/9109...
  Sentiment Score: 3
Processing tweet 333/9109...
  Sentiment Score: 3
Processing tweet 334/9109...
  Sentiment Score: 3
Processing tweet 335/9109...
  Sentiment Score: 3
Processing tweet 336/9109...
  Sentiment Score: 3
Processing tweet 337/9109...
  Sentiment Score: 3
Processing tweet 338/9109...
  Sentiment Score: 3
Processing tweet 339/9109...
  Sentiment Score: 4
Processing tweet 340/9109...
  Sentiment Score: 2
Processing tweet 341/9109...
  Sentiment Score: 4
Processing tweet 342/9109...
  Sentiment Score: 3
Processing tweet 343/9109...
  Sentiment Score: 2
Processing tweet 344/9109...
  Sentiment Score: 2
Processing tweet 345/9109...
  Sentiment Score: 3
Processing tweet 346/9109...
  Sentiment Score: 3
Processing tweet 347/9109...
  Sentiment Score: 4
Processing tweet 348/9109...
  Sentiment Score: 4
Processing tweet 349/9109...
  Sentiment Score: 4
Processing tweet 350/9109...


  Sentiment Score: 1
Processing tweet 495/9109...
  Sentiment Score: 2
Processing tweet 496/9109...
  Sentiment Score: 3
Processing tweet 497/9109...
  Sentiment Score: 3
Processing tweet 498/9109...
  Sentiment Score: 4
Processing tweet 499/9109...
  Sentiment Score: 1
Processing tweet 500/9109...
  Sentiment Score: 4
Processing tweet 501/9109...
  Sentiment Score: 3
Processing tweet 502/9109...
  Sentiment Score: 3
Processing tweet 503/9109...
  Sentiment Score: 3
Processing tweet 504/9109...
  Sentiment Score: 4
Processing tweet 505/9109...
  Sentiment Score: 4
Processing tweet 506/9109...
  Sentiment Score: 3
Processing tweet 507/9109...
  Sentiment Score: 1
Processing tweet 508/9109...
  Sentiment Score: 3
Processing tweet 509/9109...
  Sentiment Score: 4
Processing tweet 510/9109...
  Sentiment Score: 3
Processing tweet 511/9109...
  Sentiment Score: 4
Processing tweet 512/9109...
  Sentiment Score: 4
Processing tweet 513/9109...
  Sentiment Score: 2
Processing tweet 514/9109...


  Sentiment Score: 2
Processing tweet 659/9109...
  Sentiment Score: 2
Processing tweet 660/9109...
  Sentiment Score: 2
Processing tweet 661/9109...
  Sentiment Score: 4
Processing tweet 662/9109...
  Sentiment Score: 2
Processing tweet 663/9109...
  Sentiment Score: 2
Processing tweet 664/9109...
  Sentiment Score: 4
Processing tweet 665/9109...
  Sentiment Score: 2
Processing tweet 666/9109...
  Sentiment Score: 2
Processing tweet 667/9109...
  Sentiment Score: 3
Processing tweet 668/9109...
  Sentiment Score: 2
Processing tweet 669/9109...
  Sentiment Score: 3
Processing tweet 670/9109...
  Sentiment Score: 4
Processing tweet 671/9109...
  Sentiment Score: 4
Processing tweet 672/9109...
  Sentiment Score: 3
Processing tweet 673/9109...
  Sentiment Score: 3
Processing tweet 674/9109...
  Sentiment Score: 2
Processing tweet 675/9109...
  Sentiment Score: 2
Processing tweet 676/9109...
  Sentiment Score: 1
Processing tweet 677/9109...
  Sentiment Score: 1
Processing tweet 678/9109...


...

  Sentiment Score: 2
Processing tweet 8369/9109...
  Sentiment Score: 3
Processing tweet 8370/9109...
  Sentiment Score: 1
Processing tweet 8371/9109...
  Sentiment Score: 2
Processing tweet 8372/9109...
  Sentiment Score: 3
Processing tweet 8373/9109...
  Sentiment Score: 3
Processing tweet 8374/9109...
  Sentiment Score: 3
Processing tweet 8375/9109...
  Sentiment Score: 2
Processing tweet 8376/9109...
  Sentiment Score: 3
Processing tweet 8377/9109...
  Sentiment Score: 3
Processing tweet 8378/9109...
  Sentiment Score: 3
Processing tweet 8379/9109...
  Sentiment Score: 5
Processing tweet 8380/9109...
  Sentiment Score: 2
Processing tweet 8381/9109...
  Sentiment Score: 3
Processing tweet 8382/9109...
  Sentiment Score: 4
Processing tweet 8383/9109...
  Sentiment Score: 4
Processing tweet 8384/9109...
  Sentiment Score: 2
Processing tweet 8385/9109...
  Sentiment Score: 4
Processing tweet 8386/9109...
  Sentiment Score: 4
Processing tweet 8387/9109...
  Sentiment Score: 4
Processing

  Sentiment Score: 3
Processing tweet 8530/9109...
  Sentiment Score: 1
Processing tweet 8531/9109...
  Sentiment Score: 2
Processing tweet 8532/9109...
  Sentiment Score: 2
Processing tweet 8533/9109...
  Sentiment Score: 4
Processing tweet 8534/9109...
  Sentiment Score: 3
Processing tweet 8535/9109...
  Sentiment Score: 3
Processing tweet 8536/9109...
  Sentiment Score: 1
Processing tweet 8537/9109...
  Sentiment Score: 4
Processing tweet 8538/9109...
  Sentiment Score: 2
Processing tweet 8539/9109...
  Sentiment Score: 2
Processing tweet 8540/9109...
  Sentiment Score: 4
Processing tweet 8541/9109...
  Sentiment Score: 4
Processing tweet 8542/9109...
  Sentiment Score: 3
Processing tweet 8543/9109...
  Sentiment Score: 2
Processing tweet 8544/9109...
  Sentiment Score: 1
Processing tweet 8545/9109...
  Sentiment Score: 2
Processing tweet 8546/9109...
  Sentiment Score: 2
Processing tweet 8547/9109...
  Sentiment Score: 3
Processing tweet 8548/9109...
  Sentiment Score: 2
Processing

  Sentiment Score: 3
Processing tweet 8691/9109...
  Sentiment Score: 1
Processing tweet 8692/9109...
  Sentiment Score: 0
Processing tweet 8693/9109...
  Sentiment Score: 2
Processing tweet 8694/9109...
  Sentiment Score: 3
Processing tweet 8695/9109...
  Sentiment Score: 4
Processing tweet 8696/9109...
  Sentiment Score: 5
Processing tweet 8697/9109...
  Sentiment Score: 4
Processing tweet 8698/9109...
  Sentiment Score: 3
Processing tweet 8699/9109...
  Sentiment Score: 2
Processing tweet 8700/9109...
  Sentiment Score: 4
Processing tweet 8701/9109...
  Sentiment Score: 4
Processing tweet 8702/9109...
  Sentiment Score: 4
Processing tweet 8703/9109...
  Sentiment Score: 2
Processing tweet 8704/9109...
  Sentiment Score: 4
Processing tweet 8705/9109...
  Sentiment Score: 4
Processing tweet 8706/9109...
  Sentiment Score: 4
Processing tweet 8707/9109...
  Sentiment Score: 4
Processing tweet 8708/9109...
  Sentiment Score: 4
Processing tweet 8709/9109...
  Sentiment Score: 3
Processing

  Sentiment Score: 2
Processing tweet 8852/9109...
  Sentiment Score: 2
Processing tweet 8853/9109...
  Sentiment Score: 2
Processing tweet 8854/9109...
  Sentiment Score: 2
Processing tweet 8855/9109...
  Sentiment Score: 4
Processing tweet 8856/9109...
  Sentiment Score: 4
Processing tweet 8857/9109...
  Sentiment Score: 3
Processing tweet 8858/9109...
  Sentiment Score: 3
Processing tweet 8859/9109...
  Sentiment Score: 4
Processing tweet 8860/9109...
  Sentiment Score: 4
Processing tweet 8861/9109...
  Sentiment Score: 2
Processing tweet 8862/9109...
  Sentiment Score: 4
Processing tweet 8863/9109...
  Sentiment Score: 2
Processing tweet 8864/9109...
  Sentiment Score: 4
Processing tweet 8865/9109...
  Sentiment Score: 2
Processing tweet 8866/9109...
  Sentiment Score: 2
Processing tweet 8867/9109...
  Sentiment Score: 3
Processing tweet 8868/9109...
  Sentiment Score: 3
Processing tweet 8869/9109...
  Sentiment Score: 1
Processing tweet 8870/9109...
  Sentiment Score: 1
Processing

  Sentiment Score: 2
Processing tweet 9013/9109...
  Sentiment Score: 3
Processing tweet 9014/9109...
  Sentiment Score: 4
Processing tweet 9015/9109...
  Sentiment Score: 4
Processing tweet 9016/9109...
  Sentiment Score: 3
Processing tweet 9017/9109...
  Sentiment Score: 2
Processing tweet 9018/9109...
  Sentiment Score: 1
Processing tweet 9019/9109...
  Sentiment Score: 3
Processing tweet 9020/9109...
  Sentiment Score: 4
Processing tweet 9021/9109...
  Sentiment Score: 2
Processing tweet 9022/9109...
  Sentiment Score: 2
Processing tweet 9023/9109...
  Sentiment Score: 2
Processing tweet 9024/9109...
  Sentiment Score: 4
Processing tweet 9025/9109...
  Sentiment Score: 1
Processing tweet 9026/9109...
  Sentiment Score: 2
Processing tweet 9027/9109...
  Sentiment Score: 4
Processing tweet 9028/9109...
  Sentiment Score: 5
Processing tweet 9029/9109...
  Sentiment Score: 4
Processing tweet 9030/9109...
  Sentiment Score: 2
Processing tweet 9031/9109...
  Sentiment Score: 5
Processing